In [ ]:
import pandas as pd
from csae_pyutils import gsheet_to_df
from tqdm import tqdm
from apis_core.apis_entities.models import Work, Person
from apis_core.apis_metainfo.models import Collection, TempEntityClass, Uri
from apis_core.apis_relations.models import PersonWork
from apis_core.apis_vocabularies.models import PersonWorkRelation

In [ ]:
col = Collection.objects.get(id=84)
domain = "tmp-burgtheater"
person_work_relation = PersonWorkRelation.objects.get(id=1049)
col, domain, person_work_relation

In [ ]:
df = gsheet_to_df("16D-OzcZ_6NQ673TZT82hAtUrxugB9nru8B9S_uxJrPg")

In [ ]:
df.head(3)

In [ ]:
# adds tmp uris to existing works
for i, row in tqdm(df.iterrows(), total=len(df)):
    if pd.isna(row["pmb_work_id"]):
        continue
    else:
        entity = TempEntityClass.objects.get(id=int(row["pmb_work_id"]))
        tmp_url = row["work_id"]
        pmb_uri, _ = Uri.objects.get_or_create(uri=tmp_url, domain=domain, entity=entity)
        entity.collection.add()
    
    

In [ ]:
# process the rest
for i, row in tqdm(df.iterrows(), total=len(df)):
    if pd.isna(row["pmb_work_id"]):
        tmp_url = row["work_id"]
        pmb_uri, _ = Uri.objects.get_or_create(uri=tmp_url, domain=domain)
        if pmb_uri.entity:
            entity.collection.add()
            work = Work.objects.get(id=entity.id)
        else:
            work = Work.objects.create(name=row["title"])
            work.collection.add()
            pmb_uri.entity = work
            pmb_uri.save()
        if not pd.isna(row["author_pmb_ids"]):
            for x in row["author_pmb_ids"].split("|"):
                person = Person.objects.get(id=int(x))
                PersonWork.objects.get_or_create(related_person=person, related_work=work, relation_type=person_work_relation)